# 90. Monitoreo y costos

Consultas para la evidencia de monitoreo del README: el event log del pipeline (avance de cada flujo y métricas de las expectativas), el historial de corridas del job en `system.lakeflow` y el consumo en `system.billing.usage`. Si una tabla de sistema no está disponible en el workspace, la celda imprime el error para llevarlo al README y la alternativa es el historial de corridas en la interfaz del job.

In [ ]:
dbutils.widgets.text("catalogo", "electrocasa_dev")
CATALOGO = dbutils.widgets.get("catalogo").strip()
if not CATALOGO or not all(c.isalnum() or c == "_" for c in CATALOGO):
    raise ValueError("El catálogo solo admite letras, números y guion bajo")

In [ ]:
display(spark.sql(f"""
    SELECT timestamp, event_type, origin.flow_name AS flujo, message
    FROM event_log(TABLE({CATALOGO}.silver.ventas))
    WHERE event_type IN ('update_progress', 'flow_progress')
    ORDER BY timestamp DESC
    LIMIT 100
"""))

In [ ]:
display(spark.sql(f"""
    WITH expectativas AS (
      SELECT timestamp,
             origin.flow_name AS flujo,
             explode(from_json(details:flow_progress:data_quality:expectations,
                     'array<struct<name: string, dataset: string, passed_records: bigint, failed_records: bigint>>')) AS e
      FROM event_log(TABLE({CATALOGO}.silver.ventas))
      WHERE event_type = 'flow_progress' AND details:flow_progress:data_quality:expectations IS NOT NULL
    )
    SELECT e.dataset AS tabla, e.name AS expectativa,
           sum(e.passed_records) AS registros_que_pasan,
           sum(e.failed_records) AS registros_que_fallan
    FROM expectativas
    GROUP BY e.dataset, e.name
    ORDER BY tabla, expectativa
"""))

In [ ]:
def consultar(sql):
    try:
        display(spark.sql(sql))
    except Exception as error:
        print(f"No disponible: {type(error).__name__}: {error}")


consultar("""
    SELECT j.name AS job, r.run_id, r.period_start_time, r.period_end_time, r.result_state, r.trigger_type
    FROM system.lakeflow.job_run_timeline r
    JOIN (SELECT workspace_id, job_id, name,
                 row_number() OVER (PARTITION BY workspace_id, job_id ORDER BY change_time DESC) AS rn
          FROM system.lakeflow.jobs) j
      ON r.workspace_id = j.workspace_id AND r.job_id = j.job_id AND j.rn = 1
    WHERE j.name LIKE '%electrocasa%'
    ORDER BY r.period_start_time DESC
    LIMIT 50
""")

## Evidencia del conector y conteos por lote

`DESCRIBE HISTORY` de la tabla del conector: la primera corrida deja un `MERGE` con todas las filas y la segunda no agrega filas nuevas. Después, los conteos que dejó cada corrida del job, para la tabla de la sección 14.1 del README.

In [ ]:
display(spark.sql(f"DESCRIBE HISTORY {CATALOGO}.bronze.tracking_envios")
        .select("version", "timestamp", "operation", "operationMetrics"))

In [ ]:
display(spark.sql(f"""
    SELECT capa, tabla, job_run_id, filas, fecha_validacion
    FROM {CATALOGO}.default.conteos_por_corrida
    ORDER BY capa, tabla, fecha_validacion
"""))

## Calidad de Tracking (para el contrato de datos)

Variantes de estado y de courier tal como llegan del conector, fechas nulas, rechazados, eventos que quedan en Silver después de quitar los mensajes reprocesados y eventos con una venta que no existe.

In [ ]:
display(spark.sql(f"SELECT estado_entrega, count(*) AS eventos FROM {CATALOGO}.bronze.tracking_envios GROUP BY estado_entrega ORDER BY eventos DESC"))
display(spark.sql(f"SELECT courier, count(*) AS eventos FROM {CATALOGO}.bronze.tracking_envios GROUP BY courier ORDER BY eventos DESC"))
CONSULTA_TRACKING = (
    f"SELECT (SELECT count(*) FROM {CATALOGO}.bronze.tracking_envios) AS eventos_bronze, "
    f"(SELECT count(*) FROM {CATALOGO}.bronze.tracking_envios WHERE fecha_actualizacion IS NULL) AS sin_fecha_actualizacion, "
    f"(SELECT count(*) FROM {CATALOGO}.silver.cuarentena WHERE fuente = 'tracking_envios') AS rechazados, "
    f"(SELECT count(*) FROM {CATALOGO}.silver.tracking_envios) AS eventos_silver, "
    f"(SELECT count(*) FROM {CATALOGO}.silver.tracking_envios WHERE NOT venta_existente) AS venta_inexistente"
)
display(spark.sql(CONSULTA_TRACKING))

## Máscara de dni y salario

La consulta muestra si quien la corre es miembro de `electrocasa_ingenieria`. Un miembro ve los valores reales; cualquier otro usuario ve el dni con sus tres últimos dígitos y el salario nulo.

In [ ]:
display(spark.sql(
    f"SELECT is_account_group_member('electrocasa_ingenieria') AS miembro_de_ingenieria, "
    f"id_empleado, nombre, dni, salario, sucursal_id, vigente_desde "
    f"FROM {CATALOGO}.gold.empleados_vigentes ORDER BY id_empleado LIMIT 5"
))

## Respuestas de Gold

In [ ]:
CONSULTAS_GOLD = [
    f"SELECT * FROM {CATALOGO}.gold.ventas_netas_sucursal_mes ORDER BY sucursal_id, mes LIMIT 16",
    f"SELECT * FROM {CATALOGO}.gold.top_productos_unidades ORDER BY puesto",
    f"SELECT * FROM {CATALOGO}.gold.top_productos_devolucion ORDER BY puesto",
    f"SELECT fecha_corte, sum(empleados_activos) AS empleados_activos FROM {CATALOGO}.gold.dotacion_sucursal_mes GROUP BY fecha_corte ORDER BY fecha_corte",
    f"SELECT sucursal_id, sum(cantidad_ventas) AS ventas, sum(monto_ventas) AS monto FROM {CATALOGO}.gold.ventas_vendedor_mes GROUP BY sucursal_id ORDER BY ventas DESC LIMIT 10",
    f"SELECT * FROM {CATALOGO}.gold.resenas_negativas_categoria_mes ORDER BY categoria, mes",
    f"SELECT origen, sum(total_resenas) AS resenas, sum(resenas_negativas) AS negativas FROM {CATALOGO}.gold.resenas_negativas_origen_mes GROUP BY origen ORDER BY origen",
    f"SELECT * FROM {CATALOGO}.gold.despachos_courier ORDER BY courier",
]
for consulta in CONSULTAS_GOLD:
    display(spark.sql(consulta))

## Rechazos por fuente y regla

In [ ]:
display(spark.sql(
    f"SELECT fuente, regla, count(*) AS registros FROM {CATALOGO}.silver.cuarentena "
    f"LATERAL VIEW explode(reglas_fallidas) AS regla GROUP BY fuente, regla ORDER BY fuente, regla"
))

## Costos medidos (reto opcional)

Consumo del proyecto por recurso, a partir de `system.billing.usage`, filtrando por los jobs y pipelines cuyo nombre contiene `electrocasa`.

In [ ]:
consultar("""
    WITH recursos AS (
      SELECT DISTINCT job_id AS id, name, 'job' AS tipo FROM system.lakeflow.jobs WHERE name LIKE '%electrocasa%'
      UNION ALL
      SELECT DISTINCT pipeline_id AS id, name, 'pipeline' AS tipo FROM system.lakeflow.pipelines WHERE name LIKE '%electrocasa%'
    )
    SELECT r.tipo, r.name AS recurso, u.sku_name, u.usage_unit,
           round(sum(u.usage_quantity), 4) AS consumo
    FROM system.billing.usage u
    JOIN recursos r
      ON r.id = coalesce(u.usage_metadata.job_id, u.usage_metadata.dlt_pipeline_id)
    GROUP BY r.tipo, r.name, u.sku_name, u.usage_unit
    ORDER BY consumo DESC
""")